# Phase 3 — Temporal Consistency Fine-Tuning

**Goal:** Reduce HR jump between consecutive sliding windows by adding a temporal consistency (TC) loss on the 130-frame overlap region.

- Starting checkpoint: `checkpoints/phase2/best.pth` (score=1.2967)
- Training data: MCD FullHDwebcam frontal YOLO5Face cache (979 recordings)
- TC loss: NegPearson on overlap region between two consecutive 160-frame windows (stride=30)
- Exit gate: composite score does not regress from Phase 2 (1.2967) AND HR std improves

Config: `checkpoints/phase3/config.json`  
Script: `src/train_phase3.py`

In [ ]:
import json, subprocess, sys, time
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/mnt/sata-ssd/rppg_project')

with open(PROJECT_ROOT / 'checkpoints/phase3/config.json') as f:
    CFG = json.load(f)

UBFC_CACHE      = Path(CFG['ubfc_cache'])
RPPG10_CACHE    = Path(CFG['rppg10_cache'])
MCD_EVAL_CACHE  = Path(CFG['mcd_eval_cache'])
MCD_HELD_H5_DIR = Path(CFG['mcd_held_h5_dir'])
MCD_CACHE_DIR   = Path(CFG['mcd_cache_dir'])
START_CKPT      = Path(CFG['start_ckpt'])
BEST_CKPT       = Path(CFG['best_ckpt'])
METRICS_JSON    = Path(CFG['metrics_json'])

print('Config loaded.')
print(f"  start_ckpt   : {START_CKPT}")
print(f"  clip_len     : {CFG['clip_len']}  stride: {CFG['window_stride']}")
print(f"  lr_max       : {CFG['lr_max']}  lambda_tc: {CFG['lambda_tc']}")
print(f"  epochs       : {CFG['epochs']}  batch_size: {CFG['batch_size']}  grad_accum: {CFG['grad_accum_steps']}")

## 1. Cache Verification

In [ ]:
# Verify all required caches exist
checks = {
    'start_ckpt       ': START_CKPT,
    'UBFC eval cache  ': UBFC_CACHE,
    'rPPG10 eval cache': RPPG10_CACHE,
    'MCD eval cache   ': MCD_EVAL_CACHE,
    'MCD held-out h5s ': MCD_HELD_H5_DIR,
    'MCD train h5s    ': MCD_CACHE_DIR,
}
all_ok = True
for name, p in checks.items():
    if p.is_file():
        mb = p.stat().st_size / 1e6
        print(f'  {name}: OK ({mb:.0f} MB)')
    elif p.is_dir():
        n = len(list(p.glob('*.h5'))) + len(list(p.glob('*.pth')))
        print(f'  {name}: OK ({n} files)')
    else:
        print(f'  {name}: MISSING — {p}')
        all_ok = False

print('\nAll caches OK.' if all_ok else '\nERROR: missing caches above must be built first.')

## 2. Dataset Size Probe

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT / 'external' / 'FactorizePhys'))

import h5py, json, random
import pandas as pd

CLIP_LEN      = CFG['clip_len']       # 160
WINDOW_STRIDE = CFG['window_stride']  # 30
MIN_FRAMES    = CLIP_LEN + WINDOW_STRIDE + 1  # 191

with open(CFG['mcd_split_json']) as f:
    train_ids = set(json.load(f)['train'])

db = pd.read_csv(MCD_CACHE_DIR.parent / 'db.csv')
db = db[(db['patient_id'].isin(train_ids)) & (db['camera'] == 'FullHDwebcam')]
db = db[db['step'].isin(['before', 'after'])]
db = db[(db['pulse'] >= 40) & (db['pulse'] <= 160)]

n_clips, n_skip = 0, 0
for _, row in db.iterrows():
    h5_path = MCD_CACHE_DIR / f"{row.patient_id}_FullHDwebcam_{row.step}.h5"
    if not h5_path.exists():
        n_skip += 1
        continue
    try:
        with h5py.File(str(h5_path), 'r') as hf:
            n_frames = int(hf.attrs.get('n_frames', hf['frames'].shape[1]))
            ppg_std  = float(hf['ppg'][:].std())
            if ppg_std < 5.0 or n_frames < MIN_FRAMES:
                n_skip += 1
                continue
            n_clips += max(0, (n_frames - MIN_FRAMES) // CLIP_LEN + 1)
    except Exception:
        n_skip += 1

print(f'Recordings in DB (frontal, train, HR 40-160): {len(db)}')
print(f'  Skipped (missing/short/flat): {n_skip}')
print(f'  Temporal pairs:               {n_clips}')
print(f'  (stride={CLIP_LEN} → non-overlapping pair starts)')
print(f'  Min frames required:          {MIN_FRAMES}')

## 3. ClearML Task Init

In [ ]:
from clearml import Task

task = Task.init(
    project_name='rppg',
    task_name='phase3_temporal_consistency',
    tags=['phase3', 'temporal-consistency', 'mcd-frontal'],
    reuse_last_task_id=False,
)
task.connect(CFG)

TASK_ID = task.id
print(f'ClearML task ID: {TASK_ID}')

# Write task ID back to config so train_phase3.py can log scalars
CFG['clearml_task_id'] = TASK_ID
with open(PROJECT_ROOT / 'checkpoints/phase3/config.json', 'w') as f:
    json.dump(CFG, f, indent=2)
print('Task ID saved to config.')

## 4. Launch DDP Training

In [ ]:
LOG_FILE = Path('/tmp/phase3_training.log')
LOG_FILE.unlink(missing_ok=True)

cmd = [
    '/home/dex/rppg_venv/bin/torchrun',
    '--nproc_per_node=2',
    '--rdzv_backend=c10d',
    '--rdzv_endpoint=localhost:29502',
    str(PROJECT_ROOT / 'src/train_phase3.py'),
]

with open(LOG_FILE, 'w') as log_f:
    proc = subprocess.Popen(
        cmd,
        stdout=log_f,
        stderr=subprocess.STDOUT,
        cwd=str(PROJECT_ROOT),
    )
print(f'Training launched — PID {proc.pid}')
print(f'Log: {LOG_FILE}')
print('Watch with:  !tail -f /tmp/phase3_training.log')

In [ ]:
# Poll until training finishes (checks every 60s, prints last 10 lines)
import time
while proc.poll() is None:
    time.sleep(60)
    result = subprocess.run(['tail', '-n', '10', str(LOG_FILE)], capture_output=True, text=True)
    print(result.stdout, end='')
    print('---')

rc = proc.returncode
print(f'\nTraining complete — exit code {rc}')
if rc != 0:
    result = subprocess.run(['tail', '-n', '30', str(LOG_FILE)], capture_output=True, text=True)
    print(result.stdout)

## 5. Results & Plots

In [ ]:
with open(METRICS_JSON) as f:
    hist = json.load(f)

import pandas as pd
df = pd.DataFrame(hist)

# Best epoch (lowest composite score, epoch > 0)
train_df = df[df['epoch'] > 0]
best_row  = train_df.loc[train_df['score'].idxmin()]
start_row = df[df['epoch'] == 0].iloc[0]

print('Phase 3 — Metrics Summary')
print('='*60)
print(f"{'Epoch':>6} {'Loss':>7} {'UBFC':>6} {'MCD':>6} {'rPPG10':>7} {'Score':>7} {'TC_mean':>8} {'TC_med':>8}")
print('-'*60)
for _, r in df.iterrows():
    e = int(r['epoch'])
    tl = f"{r['train_loss']:.4f}" if r['train_loss'] is not None else '  —   '
    mark = ' ★' if e == int(best_row['epoch']) else ''
    print(f"{e:>6}  {tl:>7}  {r['ubfc_mae']:>5.3f}  {r['mcd_mae']:>5.3f}  "
          f"{r['rppg10_mae']:>6.3f}  {r['score']:>6.4f}  "
          f"{r['tc_mean_std']:>7.3f}  {r['tc_median_std']:>7.3f}{mark}")

print('='*60)
print(f"\nBest epoch: {int(best_row['epoch'])}  |  score={best_row['score']:.4f}")
print(f"Start: UBFC={start_row['ubfc_mae']:.3f}  MCD={start_row['mcd_mae']:.3f}  "
      f"rPPG10={start_row['rppg10_mae']:.3f}  score={start_row['score']:.4f}  "
      f"TC_mean={start_row['tc_mean_std']:.3f}")
print(f"Best:  UBFC={best_row['ubfc_mae']:.3f}  MCD={best_row['mcd_mae']:.3f}  "
      f"rPPG10={best_row['rppg10_mae']:.3f}  score={best_row['score']:.4f}  "
      f"TC_mean={best_row['tc_mean_std']:.3f}")
phase2_score = 1.2967
gate = 'PASSED' if best_row['score'] <= phase2_score else 'NOT MET'
print(f"\nExit gate (score ≤ {phase2_score}): {gate}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = df['epoch'].tolist()

# Composite score
axes[0].plot(epochs, df['score'], 'o-', color='steelblue', lw=2)
axes[0].axhline(1.0,      color='green',  ls='--', label='FP_PURE (1.0)')
axes[0].axhline(1.2967,   color='orange', ls='--', label='Phase2 best (1.2967)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Composite Score')
axes[0].set_title('Composite Score (↓ better)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Per-dataset MAE
axes[1].plot(epochs, df['ubfc_mae'],    's-', label='UBFC (×0.3/1.23)', color='royalblue')
axes[1].plot(epochs, df['mcd_mae'],     'D-', label='MCD (×0.3/12.36)', color='darkorange')
axes[1].plot(epochs, df['rppg10_mae'],  '^-', label='rPPG10 (×0.4/13.89)', color='green')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE (bpm)')
axes[1].set_title('Per-Dataset MAE (↓ better)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# HR consistency (temporal consistency metric)
axes[2].plot(epochs, df['tc_mean_std'],   'o-', label='Mean std', color='crimson')
axes[2].plot(epochs, df['tc_median_std'], 's--', label='Median std', color='salmon')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('HR std across windows (bpm)')
axes[2].set_title('HR Consistency (↓ = more stable)')
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout()
PLOT_PATH = PROJECT_ROOT / 'docs/plots/phase3_training_curves.png'
plt.savefig(str(PLOT_PATH), dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOT_PATH}')

## 6. Docs Update

In [ ]:
from datetime import date
today = date.today().isoformat()

# execution_log.md prepend
log_entry = f"""## Phase 3 — Temporal Consistency Fine-Tuning ({today})

**Script:** `src/train_phase3.py`  
**Start ckpt:** `checkpoints/phase2/best.pth` (score=1.2967)  
**Config:** `checkpoints/phase3/config.json`

| Param          | Value |
|----------------|-------|
| epochs         | {CFG['epochs']} (early stop patience={CFG['early_stop_patience']}) |
| clip_len       | {CFG['clip_len']} frames |
| window_stride  | {CFG['window_stride']} frames |
| overlap        | {CFG['clip_len'] - CFG['window_stride']} frames |
| lambda_tc      | {CFG['lambda_tc']} |
| lr_max / lr_min | {CFG['lr_max']} / {CFG['lr_min']} |
| batch_size     | {CFG['batch_size']} × 2 GPUs × grad_accum={CFG['grad_accum_steps']} = {CFG['batch_size']*2*CFG['grad_accum_steps']} effective |
| loss_cap       | {CFG['loss_cap']} |

**Results:**

| Epoch | UBFC | MCD | rPPG10 | Score | TC_mean | TC_med |
|-------|------|-----|--------|-------|---------|--------|
"""
for _, r in df.iterrows():
    e = int(r['epoch'])
    mark = ' ★' if e == int(best_row['epoch']) else ''
    log_entry += (f"| {e}{mark} | {r['ubfc_mae']:.3f} | {r['mcd_mae']:.3f} | "
                  f"{r['rppg10_mae']:.3f} | {r['score']:.4f} | "
                  f"{r['tc_mean_std']:.3f} | {r['tc_median_std']:.3f} |\n")

log_entry += f"""
**Best checkpoint:** epoch {int(best_row['epoch'])}, `checkpoints/phase3/best.pth`  
**Exit gate (score ≤ 1.2967):** {gate}  
**TC improvement:** {start_row['tc_mean_std']:.3f} → {best_row['tc_mean_std']:.3f} bpm mean HR std  
**ClearML task:** {TASK_ID}

---
"""

exec_log_path = PROJECT_ROOT / 'docs/execution_log.md'
existing = exec_log_path.read_text()
exec_log_path.write_text(log_entry + existing)
print('execution_log.md updated.')

# progress.md header
prog_path = PROJECT_ROOT / 'docs/progress.md'
prog = prog_path.read_text()
prog = prog.replace(
    'Last updated:',
    f'Last updated: {today} (Phase 3 complete — best epoch {int(best_row["epoch"])}, score={best_row["score"]:.4f}, exit gate {gate})\n\n## Archive —',
    1,
).replace(
    '## Archive —\nLast updated:',
    'Last updated:',
    1,
)
# simpler: just update header line
prog_lines = prog.split('\n')
prog_lines[2] = (f'Last updated: {today} (Phase 3 complete — '
                 f'best epoch {int(best_row["epoch"])}, score={best_row["score"]:.4f}, exit gate {gate})')
prog_path.write_text('\n'.join(prog_lines))
print('progress.md updated.')